## Append, Archive, Load Data
Purpose: To read in the new month's data, append to current database, transform into long format for BI, and archive old data for version control.

In [40]:
# All libraries
import pandas as pd
import numpy as np
import os
from typing import Iterator, Union, List, Optional
from pathlib import Path
import shutil
from datetime import date

# Today's Date for Timestamp
current_date = date.today().strftime("%Y-%m-%d")  # Format: YYYY-MM-DD; Get the current date in the desired format

# File paths
parquet_filename_p1 = f"../planning_data/database/database_cleaned_p1.parquet"
parquet_filename_p2 = f"../planning_data/database/database_cleaned_p2.parquet"
database_archive = f"../planning_data/database/archive/database_cleaned_p1_{current_date}.parquet" # Append the date to the filename
long_parquet_name = f"../planning_data/database/database_cleaned_long.parquet"
parquet_intermediates =f"../planning_data/ped_report_parquet"
parquet_intermediates_p2 =f"../planning_data/p2_sensors_parquet"
parquet_intermediate_archive =f"../planning_data/ped_report_parquet/archive"
parquet_intermediate_p2_archive =f"../planning_data/p2_sensors_parquet/archive"
calendar_filename =f"../planning_data/database/calendar.parquet"
daily_path = '../planning_data/database/daily.parquet'

In [19]:
# Read in current database, then save and timestamp old database + archive
cols = pd.read_csv("../columns.csv")["Column Names"].tolist()
cols_p2 = pd.read_csv("../p2_columns.csv")["Column Names"].tolist()
df_database_p1 = pd.read_parquet(parquet_filename_p1,columns= cols, engine="pyarrow")
df_database_p1.to_parquet(database_archive, engine="pyarrow", index=False)

df_database_p1

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z27_T3TransitLobby_Stair50_DOWN_IN,Z27_T3TransitLobby_Stair50_UP_OUT,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV
0,2020-02-24 00:00:00,2020-02-24 00:05:00,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0
1,2020-02-24 00:05:00,2020-02-24 00:10:00,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2020-02-24 00:10:00,2020-02-24 00:15:00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
3,2020-02-24 00:15:00,2020-02-24 00:20:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,2020-02-24 00:20:00,2020-02-24 00:25:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536539,2025-03-31 23:35:00,2025-03-31 23:40:00,0.0,2.0,1.0,2.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0
536540,2025-03-31 23:40:00,2025-03-31 23:45:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0
536541,2025-03-31 23:45:00,2025-03-31 23:50:00,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536542,2025-03-31 23:50:00,2025-03-31 23:55:00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,...,0.0,0.0,1.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0


In [15]:
# Process all new files and combine them all.
dfs = []
parquet_files = [f for f in os.listdir(parquet_intermediates) if f.endswith('.parquet')]

for file in parquet_files:
    file_path = os.path.join(parquet_intermediates, file)
    arch_path = os.path.join(parquet_intermediate_archive, file)
    try:
        df = pd.read_parquet(file_path, engine="pyarrow")
        print(f"Successfully read: {file}")
        dfs.append(df)
        print(f"Successfully appended: {file}")
        dest = shutil.move(file_path, arch_path) 
    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")
if dfs:
    # Concatenate all DataFrames into one
    combined_p1_df = pd.concat(dfs, ignore_index=True)
else:
    print("No Parquet files to process.")

No valid data to save.
No Parquet files to process.


In [16]:
# Remove data dumps and replace them with NA
if dfs:
    df = combined_p1_df\
        .drop_duplicates(subset=['From'], keep='first')\
        .reset_index(drop = True)
    df.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)

In [17]:
# Append if new data added
if dfs:
    re_concatenate = [df,df_database_p1]
else:
    re_concatenate = [df_database_p1]

combined_p1_df = pd.concat(re_concatenate, ignore_index=True)
combined_p1_df = combined_p1_df.drop_duplicates().sort_values(by = "From")\
                .reset_index(drop = True)
combined_p1_df

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z27_T3TransitLobby_Stair50_DOWN_IN,Z27_T3TransitLobby_Stair50_UP_OUT,Z28_T3Elev-C1_OUTOFELEV_IN,Z28_T3Elev-C1_INTOELEV_OUT,Z28_1Train-C1-SConc_AllDoors_IN,Z28_1Train-C1-SConc_AllDoors_OUT,Z31_T3Elev-C2_OUTOFELEV_IN,Z31_T3Elev-C2_INTOELEV_OUT,Z29_C2_SWOculus_HubElev_OUTOFELEV,Z29_C2_SWOculus_HubElev_INTOELEV
0,2020-02-24 00:00:00,2020-02-24 00:05:00,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,0.0,0.0,0.0,0.0,4.0,0.0,0.0,1.0,0.0
1,2020-02-24 00:05:00,2020-02-24 00:10:00,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,2020-02-24 00:10:00,2020-02-24 00:15:00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
3,2020-02-24 00:15:00,2020-02-24 00:20:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,0.0
4,2020-02-24 00:20:00,2020-02-24 00:25:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536539,2025-03-31 23:35:00,2025-03-31 23:40:00,0.0,2.0,1.0,2.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,3.0,0.0,0.0,1.0,0.0
536540,2025-03-31 23:40:00,2025-03-31 23:45:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,3.0
536541,2025-03-31 23:45:00,2025-03-31 23:50:00,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
536542,2025-03-31 23:50:00,2025-03-31 23:55:00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,...,0.0,0.0,1.0,1.0,1.0,2.0,0.0,0.0,0.0,0.0


In [20]:
# Save new database
combined_p1_df.to_parquet(parquet_filename_p1, engine="pyarrow", index=False)

In [10]:
# Process all new files and combine them all.
dfs = []
parquet_files = [f for f in os.listdir(parquet_intermediates_p2) if f.endswith('.parquet')]

for file in parquet_files:
    file_path = os.path.join(parquet_intermediates_p2, file)
    arch_path = os.path.join(parquet_intermediate_p2_archive, file)
    try:
        df = pd.read_parquet(file_path, engine="pyarrow")
        print(f"Successfully read: {file}")
        dfs.append(df)
        print(f"Successfully appended: {file}")
        dest = shutil.move(file_path, arch_path) 
    except Exception as e:
        print(f"Error reading {file}: {e}")
else:
    print("No valid data to save.")
if dfs:
    # Concatenate all DataFrames into one
    combined_p2_df = pd.concat(dfs, ignore_index=True)
else:
    print("No Parquet files to process.")

Successfully read: 2023-10.parquet
Successfully appended: 2023-10.parquet
Successfully read: 2023-11.parquet
Successfully appended: 2023-11.parquet
Successfully read: 2023-12.parquet
Successfully appended: 2023-12.parquet
Successfully read: 2024-01.parquet
Successfully appended: 2024-01.parquet
Successfully read: 2024-02.parquet
Successfully appended: 2024-02.parquet
Successfully read: 2024-03.parquet
Successfully appended: 2024-03.parquet
Successfully read: 2024-04.parquet
Successfully appended: 2024-04.parquet
Successfully read: 2024-05.parquet
Successfully appended: 2024-05.parquet
Successfully read: 2024-06.parquet
Successfully appended: 2024-06.parquet
Successfully read: 2024-07.parquet
Successfully appended: 2024-07.parquet
Successfully read: 2024-08.parquet
Successfully appended: 2024-08.parquet
Successfully read: 2024-09.parquet
Successfully appended: 2024-09.parquet
Successfully read: 2024-10.parquet
Successfully appended: 2024-10.parquet
Successfully read: 2024-11.parquet
Suc

In [12]:
# Remove data dumps and replace them with NA
if dfs:
    df = combined_p2_df\
        .drop_duplicates(subset=['From'], keep='first')\
        .reset_index(drop = True)
    df.iloc[:, 2:] = df.iloc[:, 2:].mask(df.iloc[:, 2:] > 500, np.nan)
combined_p2_df

,From,To Time,Z08_HEC1M-ELEV18_IN,Z08_HEC1M-ELEV18_OUT,Z33_C1-WestConcourse_IN,Z33_C1-WestConcourse_OUT,Z32_T1ESC19&20/ZONE32_NorthEsc_IN,Z32_T1ESC19&20/ZONE32_NorthEsc_OUT,Z32_T1ESC19&20/ZONE32_SouththEsc_IN,Z32_T1ESC19&20/ZONE32_SouththEsc_OUT,...,Z19_PATHturnstiles-East_249-255(south)_IN,Z19_PATHturnstiles-East_249-255(south)_OUT,Z19_PATHturnstiles-East_256-261(center)_IN,Z19_PATHturnstiles-East_256-261(center)_OUT,Z19_PATHturnstiles-East_262-267(center)_IN,Z19_PATHturnstiles-East_262-267(center)_OUT,Z19_PATHturnstiles-East_268-274(north)_IN,Z19_PATHturnstiles-East_268-274(north)_OUT,Z20_PATHturnstiles-SE_Passageway-North_NB,Z20_PATHturnstiles-SE_Passageway-North_SB
0,2023-10-01 00:00:00,2023-10-01 00:05:00,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,17.0,1.0,13.0,1.0,9.0,0.0,7.0,3.0,0.0
1,2023-10-01 00:05:00,2023-10-01 00:10:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,14.0,0.0,9.0,1.0,12.0,0.0,18.0,5.0,3.0
2,2023-10-01 00:10:00,2023-10-01 00:15:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,18.0,0.0,1.0,5.0,16.0,2.0,3.0
3,2023-10-01 00:15:00,2023-10-01 00:20:00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,26.0,18.0,47.0,21.0,19.0,12.0,3.0,23.0,5.0,0.0
4,2023-10-01 00:20:00,2023-10-01 00:25:00,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,3.0,17.0,1.0,8.0,0.0,11.0,1.0,14.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131899,2024-12-31 23:35:00,2024-12-31 23:40:00,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,2.0,16.0,0.0,8.0,0.0,3.0,0.0,8.0,6.0,1.0
131900,2024-12-31 23:40:00,2024-12-31 23:45:00,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,7.0,23.0,2.0,16.0,1.0,11.0,7.0,3.0,7.0,0.0
131901,2024-12-31 23:45:00,2024-12-31 23:50:00,0.0,0.0,1.0,2.0,0.0,0.0,0.0,0.0,...,21.0,15.0,5.0,9.0,7.0,16.0,17.0,5.0,3.0,2.0
131902,2024-12-31 23:50:00,2024-12-31 23:55:00,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,10.0,0.0,10.0,0.0,2.0,0.0,0.0,3.0,0.0


In [22]:
# Save new database
combined_p2_df.to_parquet(parquet_filename_p2, engine="pyarrow", index=False)

In [35]:
different_cols = combined_p2_df.columns.difference(combined_p1_df.columns)

combined_p2_df = combined_p2_df[different_cols]

# different_cols.size
combined_df = pd.merge(combined_p1_df, combined_p2_df, how='left', left_index=True,
                  right_index=True)

combined_df

,From,To Time,Z01_T4-ChurchSt_RevDoor_IN,Z01_T4-ChurchSt_RevDoor_OUT,Z01_T4-ChurchSt_SwingDoor_IN,Z01_T4-ChurchSt_SwingDoor_OUT,Z02_T4-LibertySt_EastEsc46_IN,Z02_T4-LibertySt_EastEsc46_OUT,Z02_T4-LibertySt_WestEsc45_IN,Z02_T4-LibertySt_WestEsc45_OUT,...,Z41_SouthWestPassage(South)_IN,Z41_SouthWestPassage(South)_OUT,Z42_CenterPassage_IN,Z42_CenterPassage_OUT,Z42_CenterPassage_Stair_IN,Z42_CenterPassage_Stair_OUT,Z43_NorthWestPassage_IN,Z43_NorthWestPassage_OUT,Z43_SouthWestPassage_IN,Z43_SouthWestPassage_OUT
0,2020-02-24 00:00:00,2020-02-24 00:05:00,1.0,0.0,0.0,0.0,0.0,1.0,3.0,0.0,...,0.0,4.0,24.0,0.0,0.0,0.0,0.0,0.0,3.0,10.0
1,2020-02-24 00:05:00,2020-02-24 00:10:00,0.0,0.0,3.0,0.0,0.0,1.0,4.0,0.0,...,0.0,1.0,11.0,0.0,0.0,0.0,1.0,0.0,0.0,6.0
2,2020-02-24 00:10:00,2020-02-24 00:15:00,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0,...,0.0,0.0,10.0,0.0,0.0,0.0,0.0,0.0,1.0,2.0
3,2020-02-24 00:15:00,2020-02-24 00:20:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,22.0,0.0,0.0,0.0,0.0,0.0,1.0,4.0
4,2020-02-24 00:20:00,2020-02-24 00:25:00,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,10.0,11.0,0.0,0.0,0.0,0.0,0.0,6.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
536539,2025-03-31 23:35:00,2025-03-31 23:40:00,0.0,2.0,1.0,2.0,4.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536540,2025-03-31 23:40:00,2025-03-31 23:45:00,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536541,2025-03-31 23:45:00,2025-03-31 23:50:00,0.0,0.0,0.0,0.0,4.0,0.0,0.0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
536542,2025-03-31 23:50:00,2025-03-31 23:55:00,0.0,1.0,0.0,0.0,1.0,0.0,0.0,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [36]:
# Combine the first two columns with the filtered data
combined_df["Time_Bucket"] = combined_df["From"].dt.floor("30T")  # "30T" means 30-minute interval
grouped_df = combined_df.groupby("Time_Bucket").sum(numeric_only=True).reset_index()
long_df = grouped_df.melt(id_vars=["Time_Bucket"], var_name="Location", value_name="Count")
long_df['date'] = long_df['Time_Bucket'].dt.date#.astype('datetime64[ns]')
long_df['time'] = long_df['Time_Bucket'].dt.time
long_df = long_df.drop(columns=['Time_Bucket'])
long_df.to_parquet(long_parquet_name, engine="pyarrow", index=False)
long_df

C:\Users\schew\AppData\Local\Temp\ipykernel_11328\4121102928.py:2: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  combined_df["Time_Bucket"] = combined_df["From"].dt.floor("30T")  # "30T" means 30-minute interval


,Location,Count,date,time
0,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,00:00:00
1,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,00:30:00
2,Z01_T4-ChurchSt_RevDoor_IN,0.0,2020-02-24,01:00:00
3,Z01_T4-ChurchSt_RevDoor_IN,2.0,2020-02-24,01:30:00
4,Z01_T4-ChurchSt_RevDoor_IN,1.0,2020-02-24,02:00:00
...,...,...,...,...
23250235,Z43_SouthWestPassage_OUT,0.0,2025-03-31,21:30:00
23250236,Z43_SouthWestPassage_OUT,0.0,2025-03-31,22:00:00
23250237,Z43_SouthWestPassage_OUT,0.0,2025-03-31,22:30:00
23250238,Z43_SouthWestPassage_OUT,0.0,2025-03-31,23:00:00


In [37]:
# daily_path
daily_db = pd.read_parquet(daily_path)
daily_db

,From,Location,Count
0,2019-01-01,Z02_T4-LibertySt_EastEsc46_IN,9870.0
1,2019-01-02,Z02_T4-LibertySt_EastEsc46_IN,5697.0
2,2019-01-03,Z02_T4-LibertySt_EastEsc46_IN,6187.0
3,2019-01-04,Z02_T4-LibertySt_EastEsc46_IN,2916.0
4,2019-01-05,Z02_T4-LibertySt_EastEsc46_IN,748.0
...,...,...,...
143984,2025-03-31,Z28_T3Elev-C1_OUTOFELEV_IN,242.0
143985,2025-03-31,Z29_1Train-C2-SWOculus_AllDoors_IN,2409.0
143986,2025-03-31,Z30_1Train-C2-NWOculus_EastSwingDoors_IN,0.0
143987,2025-03-31,Z30_1Train-C2-NWOculus_West2SwingDoors_IN,0.0


In [38]:
# Convert 'Time_Bucket' to datetime

# Get the min/max dates
min_date = daily_db['From'].min()
max_date = daily_db["From"].max()

# Generate date range
date_range = pd.date_range(start=min_date, end=max_date)

# Create DataFrame
df_calendar = pd.DataFrame({
    "date_id": date_range.date,
    "week_of_year": date_range.isocalendar().week,
    "day_of_year": date_range.dayofyear,
    "day_of_week": date_range.strftime('%A'),
    "day_of_week_num": date_range.weekday,
    "month_num": date_range.month,
    "day_type": ['Weekend' if d.weekday() in [5, 6] else 'Weekday' for d in date_range],
    "month_name": date_range.strftime('%B'),
    "month_short": date_range.strftime('%b'),
    "quarter": date_range.quarter,
    "year": date_range.year,
    "year_quarter": [f"{y}-Q{q}" for y, q in zip(date_range.year, date_range.quarter)],
    "month_year": date_range.strftime('%b %Y')
}).reset_index(drop = True)
df_calendar.to_parquet(calendar_filename, engine="pyarrow", index=False)
df_calendar

,date_id,week_of_year,day_of_year,day_of_week,day_of_week_num,month_num,day_type,month_name,month_short,quarter,year,year_quarter,month_year
0,2019-01-01,1,1,Tuesday,1,1,Weekday,January,Jan,1,2019,2019-Q1,Jan 2019
1,2019-01-02,1,2,Wednesday,2,1,Weekday,January,Jan,1,2019,2019-Q1,Jan 2019
2,2019-01-03,1,3,Thursday,3,1,Weekday,January,Jan,1,2019,2019-Q1,Jan 2019
3,2019-01-04,1,4,Friday,4,1,Weekday,January,Jan,1,2019,2019-Q1,Jan 2019
4,2019-01-05,1,5,Saturday,5,1,Weekend,January,Jan,1,2019,2019-Q1,Jan 2019
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2277,2025-03-27,13,86,Thursday,3,3,Weekday,March,Mar,1,2025,2025-Q1,Mar 2025
2278,2025-03-28,13,87,Friday,4,3,Weekday,March,Mar,1,2025,2025-Q1,Mar 2025
2279,2025-03-29,13,88,Saturday,5,3,Weekend,March,Mar,1,2025,2025-Q1,Mar 2025
2280,2025-03-30,13,89,Sunday,6,3,Weekend,March,Mar,1,2025,2025-Q1,Mar 2025


In [39]:
print(date_range)

DatetimeIndex(['2019-01-01', '2019-01-02', '2019-01-03', '2019-01-04',
               '2019-01-05', '2019-01-06', '2019-01-07', '2019-01-08',
               '2019-01-09', '2019-01-10',
               ...
               '2025-03-22', '2025-03-23', '2025-03-24', '2025-03-25',
               '2025-03-26', '2025-03-27', '2025-03-28', '2025-03-29',
               '2025-03-30', '2025-03-31'],
              dtype='datetime64[ns]', length=2282, freq='D')
